# test_registry.ipynb：逐行演算 `data/loaders/registry.py`

这个 notebook 对应 `test_prosumer.ipynb` 的下一层：`prosumer.py` 负责读取、对齐和切 episode；`registry.py` 负责把配置对象翻译成 `ProsumerDataset` 的构造参数。

每个代码 cell 都采用同一种格式：

1. 给一个具体的输入例子。
2. 按源码关键语句逐句展开中间结果。
3. 调用真实函数，展示最终输出并用断言校验。

In [1]:
from pathlib import Path
import sys
from tempfile import TemporaryDirectory

import pandas as pd
from IPython.display import display


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "data").is_dir() and (candidate / "tests").is_dir():
            return candidate
    raise RuntimeError(f"Cannot locate project root from {start}")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from data.loaders.prosumer import DEFAULT_PROCESSED_SUBDIR, ProsumerDataset
from data.loaders.registry import (
    _USE_CFG_VALUE,
    _normalized_optional_date,
    _resolve_split_dates,
    build_dataset,
    resolve_dataset_window_spec,
    resolve_test_episode_limit,
    resolve_train_episode_limit,
)
from tests.support.helpers import make_smoke_config

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 140)


def show_steps(title, input_text, rows, final_output=None):
    print()
    print(title)
    print(f"输入：{input_text}")
    display(pd.DataFrame(rows, columns=["步骤", "对应源码/表达式", "怎么算", "结果"]))
    if final_output is not None:
        print("最终输出：")
        display(final_output)


def make_registry_demo_cfg(temp_dir, *, episode_limit=4, train_window_days=2, window_stride_days=1):
    cfg = make_smoke_config(temp_dir, algorithm="MADDPG")
    cfg.env.episode_limit = int(episode_limit)
    cfg.env.train_window_days = int(train_window_days)
    cfg.env.window_stride_days = int(window_stride_days)
    return cfg


rows = [
    [1, "PROJECT_ROOT = find_project_root(Path.cwd().resolve())", "从当前 notebook 工作目录向上找包含 data/ 和 tests/ 的目录。", str(PROJECT_ROOT)],
    [2, "sys.path.insert(0, str(PROJECT_ROOT))", "把项目根目录加入 import 搜索路径，避免从 tests/testnotebook 启动时找不到 data 包。", str(PROJECT_ROOT) in sys.path],
    [3, "from data.loaders.registry import ...", "导入本 notebook 要逐行演算的 registry 函数。", "导入成功"],
    [4, "make_registry_demo_cfg(temp_dir, episode_limit=4, train_window_days=2)", "后续每节都用同一类数学例子：基础 episode=4 步，训练窗口=2 天，所以 train episode=4*2=8 步。", "helper 已定义"],
]
show_steps("0. 准备环境和公共 helper", f"cwd={Path.cwd().resolve()}", rows, {"project_root": str(PROJECT_ROOT), "processed_subdir": str(DEFAULT_PROCESSED_SUBDIR)})


0. 准备环境和公共 helper
输入：cwd=D:\GithubProject\MADRL_ESS\tests\testnotebook


,步骤,对应源码/表达式,怎么算,结果
0,1,PROJECT_ROOT = find_project_root(Path.cwd().resolve()),从当前 notebook 工作目录向上找包含 data/ 和 tests/ 的目录。,D:\GithubProject\MADRL_ESS
1,2,"sys.path.insert(0, str(PROJECT_ROOT))",把项目根目录加入 import 搜索路径，避免从 tests/testnotebook 启动时找不到 data 包。,True
2,3,from data.loaders.registry import ...,导入本 notebook 要逐行演算的 registry 函数。,导入成功
3,4,"make_registry_demo_cfg(temp_dir, episode_limit=4, train_window_days=2)",后续每节都用同一类数学例子：基础 episode=4 步，训练窗口=2 天，所以 train episode=4*2=8 步。,helper 已定义


最终输出：


{'project_root': 'D:\\GithubProject\\MADRL_ESS',
 'processed_subdir': 'processed\\prosumer'}

## 1. `_normalized_optional_date(value)`：统一日期边界

这个小函数负责把配置中的可选日期整理成统一形态：

- `None` 表示没有日期边界。
- 空字符串 `""` 也被当作没有日期边界。
- 普通字符串会先 `strip()`，再原样返回。

具体例子：`" 2020-01-05 " -> "2020-01-05"`，`"" -> None`。

In [2]:
examples = [" 2020-01-05 ", "", None]
actual = {repr(value): _normalized_optional_date(value) for value in examples}

rows = [
    [1, "if value is None", "value=' 2020-01-05 '，不是 None，继续往下。", False],
    [2, "text = str(value).strip()", "去掉首尾空格：str(' 2020-01-05 ').strip()", "2020-01-05"],
    [3, "return None if text == '' else text", "text 非空，所以返回日期字符串。", _normalized_optional_date(" 2020-01-05 ")],
    [4, "if value is None", "value=''，不是 None，继续往下。", False],
    [5, "text = str(value).strip()", "空字符串 strip 后仍是空字符串。", ""],
    [6, "return None if text == '' else text", "text == '' 成立，所以返回 None。", _normalized_optional_date("")],
    [7, "if value is None: return None", "value=None，第一句直接命中，后面的 strip 不执行。", _normalized_optional_date(None)],
]
show_steps("1. _normalized_optional_date 的逐句演算", "values=[' 2020-01-05 ', '', None]", rows, actual)

assert actual["' 2020-01-05 '"] == "2020-01-05"
assert actual["''"] is None
assert actual["None"] is None


1. _normalized_optional_date 的逐句演算
输入：values=[' 2020-01-05 ', '', None]


,步骤,对应源码/表达式,怎么算,结果
0,1,if value is None,value=' 2020-01-05 '，不是 None，继续往下。,False
1,2,text = str(value).strip(),去掉首尾空格：str(' 2020-01-05 ').strip(),2020-01-05
2,3,return None if text == '' else text,text 非空，所以返回日期字符串。,2020-01-05
3,4,if value is None,value=''，不是 None，继续往下。,False
4,5,text = str(value).strip(),空字符串 strip 后仍是空字符串。,
5,6,return None if text == '' else text,text == '' 成立，所以返回 None。,None
6,7,if value is None: return None,value=None，第一句直接命中，后面的 strip 不执行。,None


最终输出：


{"' 2020-01-05 '": '2020-01-05', "''": None, 'None': None}

## 2. `_resolve_split_dates(cfg, mode, override...)`：选择 train/test 年份和日期

这个函数把 `cfg.data` 中的 train/test 年份、日期边界解析出来。

这里的关键点是 `_USE_CFG_VALUE = object()`：它是一个唯一的哨兵值，用来区分“调用方没传 override”和“调用方明确传了 None”。


In [3]:
with TemporaryDirectory() as temp_dir:
    cfg = make_registry_demo_cfg(temp_dir)
    cfg.data.train_start_date = "2019-01-05"
    cfg.data.train_end_date = ""
    cfg.data.test_start_date = "2020-04-01"
    cfg.data.test_end_date = "2020-04-15"

    train_dates = _resolve_split_dates(cfg, "train")
    test_override_dates = _resolve_split_dates(
        cfg,
        "test",
        override_start_date=None,
        override_end_date="2020-01-02",
    )

rows = [
    [1, "selected_year = cfg.data.train_year if mode == 'train' else cfg.data.test_year", "mode='train'，所以选 train_year。", train_dates[0]],
    [2, "start_date = _normalized_optional_date(cfg.data.train_start_date)", "cfg.data.train_start_date='2019-01-05'，非空，返回原日期字符串。", train_dates[1]],
    [3, "end_date = _normalized_optional_date(cfg.data.train_end_date)", "cfg.data.train_end_date=''，归一化为 None。", train_dates[2]],
    [4, "override_start_date is not _USE_CFG_VALUE", "train 调用没有传 override，所以仍是哨兵值，不覆盖 cfg 的值。", False],
    [5, "selected_year = cfg.data.test_year", "mode='test'，所以选 test_year。", test_override_dates[0]],
    [6, "override_start_date=None", "None 不是哨兵值，表示调用方明确要把 start_date 覆盖成 None。", test_override_dates[1]],
    [7, "override_end_date='2020-01-02'", "传入具体日期，覆盖 cfg.data.test_end_date='2020-04-15'。", test_override_dates[2]],
]
show_steps(
    "2. _resolve_split_dates 的逐句演算",
    "train_start='2019-01-05', train_end='', test override start=None, end='2020-01-02'",
    rows,
    {"train_dates": train_dates, "test_override_dates": test_override_dates, "sentinel_type": type(_USE_CFG_VALUE).__name__},
)

assert train_dates == (2019, "2019-01-05", None)
assert test_override_dates == (2020, None, "2020-01-02")


2. _resolve_split_dates 的逐句演算
输入：train_start='2019-01-05', train_end='', test override start=None, end='2020-01-02'


,步骤,对应源码/表达式,怎么算,结果
0,1,selected_year = cfg.data.train_year if mode == 'train' else cfg.data.test_year,mode='train'，所以选 train_year。,2019
1,2,start_date = _normalized_optional_date(cfg.data.train_start_date),cfg.data.train_start_date='2019-01-05'，非空，返回原日期字符串。,2019-01-05
2,3,end_date = _normalized_optional_date(cfg.data.train_end_date),cfg.data.train_end_date=''，归一化为 None。,None
3,4,override_start_date is not _USE_CFG_VALUE,train 调用没有传 override，所以仍是哨兵值，不覆盖 cfg 的值。,False
4,5,selected_year = cfg.data.test_year,mode='test'，所以选 test_year。,2020
5,6,override_start_date=None,None 不是哨兵值，表示调用方明确要把 start_date 覆盖成 None。,None
6,7,override_end_date='2020-01-02',传入具体日期，覆盖 cfg.data.test_end_date='2020-04-15'。,2020-01-02


最终输出：


{'train_dates': (2019, '2019-01-05', None),
 'test_override_dates': (2020, None, '2020-01-02'),
 'sentinel_type': 'object'}

## 3. `resolve_dataset_window_spec(cfg, mode)`：把窗口配置变成数学长度

这是 `registry.py` 里最核心的纯配置计算。

具体数学例子：

- `cfg.env.episode_limit = 4`
- `cfg.env.train_window_days = 2`
- `cfg.env.window_stride_days = 1`

那么训练窗口长度是 `4 * 2 = 8`，训练窗口步长是 `4 * 1 = 4`；测试集不滚动多日窗口，所以测试长度仍是 `4`。

In [4]:
with TemporaryDirectory() as temp_dir:
    cfg = make_registry_demo_cfg(temp_dir, episode_limit=4, train_window_days=2, window_stride_days=1)
    train_window = resolve_dataset_window_spec(cfg, "train")
    test_window = resolve_dataset_window_spec(cfg, "test")

rows = [
    [1, "base_episode_length = int(cfg.env.episode_limit)", "基础 episode 长度来自 cfg.env.episode_limit。", 4],
    [2, "if mode == 'train'", "mode='train'，进入训练窗口分支。", True],
    [3, "train_window_days = int(cfg.env.train_window_days)", "训练窗口覆盖 2 个基础 episode/day。", 2],
    [4, "window_stride_days = int(cfg.env.window_stride_days)", "相邻训练窗口每次移动 1 个基础 episode/day。", 1],
    [5, "episode_length = base_episode_length * train_window_days", "4 * 2 = 8。", train_window["episode_length"]],
    [6, "window_stride_steps = base_episode_length * window_stride_days", "4 * 1 = 4。", train_window["window_stride_steps"]],
    [7, "window_strategy = 'rolling_window'", "训练集使用滚动窗口。", train_window["window_strategy"]],
    [8, "mode != 'train'", "mode='test' 时走测试分支。", True],
    [9, "test episode_length = base_episode_length", "测试集保持单个基础 episode 长度：4。", test_window["episode_length"]],
    [10, "test window_strategy = 'cfg_window'", "测试集使用配置窗口，不做多日 rolling。", test_window["window_strategy"]],
]
show_steps(
    "3. resolve_dataset_window_spec 的逐句演算",
    "episode_limit=4, train_window_days=2, window_stride_days=1",
    rows,
    {"train_window": train_window, "test_window": test_window},
)

assert train_window == {
    "episode_length": 8,
    "window_stride_steps": 4,
    "window_strategy": "rolling_window",
    "window_days": 2,
    "window_stride_days": 1,
    "base_episode_length": 4,
}
assert test_window == {
    "episode_length": 4,
    "window_stride_steps": 4,
    "window_strategy": "cfg_window",
    "window_days": 1,
    "window_stride_days": 1,
    "base_episode_length": 4,
}


3. resolve_dataset_window_spec 的逐句演算
输入：episode_limit=4, train_window_days=2, window_stride_days=1


,步骤,对应源码/表达式,怎么算,结果
0,1,base_episode_length = int(cfg.env.episode_limit),基础 episode 长度来自 cfg.env.episode_limit。,4
1,2,if mode == 'train',mode='train'，进入训练窗口分支。,True
2,3,train_window_days = int(cfg.env.train_window_days),训练窗口覆盖 2 个基础 episode/day。,2
3,4,window_stride_days = int(cfg.env.window_stride_days),相邻训练窗口每次移动 1 个基础 episode/day。,1
4,5,episode_length = base_episode_length * train_window_days,4 * 2 = 8。,8
5,6,window_stride_steps = base_episode_length * window_stride_days,4 * 1 = 4。,4
6,7,window_strategy = 'rolling_window',训练集使用滚动窗口。,rolling_window
7,8,mode != 'train',mode='test' 时走测试分支。,True
8,9,test episode_length = base_episode_length,测试集保持单个基础 episode 长度：4。,4
9,10,test window_strategy = 'cfg_window',测试集使用配置窗口，不做多日 rolling。,cfg_window


最终输出：


{'train_window': {'episode_length': 8,
  'window_stride_steps': 4,
  'window_strategy': 'rolling_window',
  'window_days': 2,
  'window_stride_days': 1,
  'base_episode_length': 4},
 'test_window': {'episode_length': 4,
  'window_stride_steps': 4,
  'window_strategy': 'cfg_window',
  'window_days': 1,
  'window_stride_days': 1,
  'base_episode_length': 4}}

## 4. `resolve_train_episode_limit` / `resolve_test_episode_limit`：给外部模块的统一口径

这两个函数只是语义包装器，但很有用：外部训练、checkpoint、shared-data 不需要自己知道 train/test window 规则，只拿最终 episode 长度即可。

In [5]:
with TemporaryDirectory() as temp_dir:
    cfg = make_registry_demo_cfg(temp_dir, episode_limit=4, train_window_days=2, window_stride_days=1)
    train_episode_limit = resolve_train_episode_limit(cfg)
    test_episode_limit = resolve_test_episode_limit(cfg)

rows = [
    [1, "resolve_train_episode_limit(cfg)", "内部调用 resolve_dataset_window_spec(cfg, 'train')。", "train window spec"],
    [2, "['episode_length']", "从 train window spec 取 episode_length，也就是 4 * 2。", train_episode_limit],
    [3, "int(...)", "确保返回 Python int，方便训练步数、manifest、checkpoint 使用。", type(train_episode_limit).__name__],
    [4, "resolve_test_episode_limit(cfg)", "内部调用 resolve_dataset_window_spec(cfg, 'test')。", "test window spec"],
    [5, "['episode_length']", "测试集 episode_length 是基础长度 4。", test_episode_limit],
]
show_steps(
    "4. resolve_train_episode_limit / resolve_test_episode_limit 的逐句演算",
    "episode_limit=4, train_window_days=2，因此 train=8, test=4",
    rows,
    {"train_episode_limit": train_episode_limit, "test_episode_limit": test_episode_limit},
)

assert train_episode_limit == 8
assert test_episode_limit == 4


4. resolve_train_episode_limit / resolve_test_episode_limit 的逐句演算
输入：episode_limit=4, train_window_days=2，因此 train=8, test=4


,步骤,对应源码/表达式,怎么算,结果
0,1,resolve_train_episode_limit(cfg),"内部调用 resolve_dataset_window_spec(cfg, 'train')。",train window spec
1,2,['episode_length'],从 train window spec 取 episode_length，也就是 4 * 2。,8
2,3,int(...),确保返回 Python int，方便训练步数、manifest、checkpoint 使用。,int
3,4,resolve_test_episode_limit(cfg),"内部调用 resolve_dataset_window_spec(cfg, 'test')。",test window spec
4,5,['episode_length'],测试集 episode_length 是基础长度 4。,4


最终输出：


{'train_episode_limit': 8, 'test_episode_limit': 4}

## 5. `build_dataset(cfg, mode='train')`：把配置翻译成 `ProsumerDataset(...)`

`build_dataset` 是这个文件的主入口。它不亲自读取 CSV；它把 `cfg` 里的路径、日期、窗口、agent、PV/负荷配置整理好，然后交给 `ProsumerDataset`。

In [6]:
with TemporaryDirectory() as temp_dir:
    cfg = make_registry_demo_cfg(temp_dir, episode_limit=4, train_window_days=2, window_stride_days=1)
    dataset = build_dataset(cfg, mode="train")
    episode = dataset.get_episode(0)
    signals = episode["signals"]

    expected_data_dir = Path(cfg.data.data_dir) / DEFAULT_PROCESSED_SUBDIR
    window_spec = resolve_dataset_window_spec(cfg, "train")
    split_dates = _resolve_split_dates(cfg, "train")

    rows = [
        [1, "data_dir = Path(cfg.data.data_dir or default_data)", "cfg.data.data_dir 已由 make_smoke_config 指向临时 data 目录。", str(Path(cfg.data.data_dir))],
        [2, "selected_year, start_date, end_date = _resolve_split_dates(cfg, 'train')", "训练 split 选择 train_year；默认没有显式日期边界。", split_dates],
        [3, "window_spec = resolve_dataset_window_spec(cfg, 'train')", "复用第 3 节数学：episode_length=4*2=8, stride=4*1=4。", window_spec],
        [4, "history_warmup_steps = forecast.history_window if forecast.type == 'lstm' else 0", f"当前 forecast.type={cfg.forecast.type!r}，不是 LSTM 主线则为 0。", dataset.history_warmup_steps],
        [5, "ProsumerDataset(data_dir=..., episode_length=..., n_agents=...)", "把所有 cfg 字段整理成 ProsumerDataset 构造参数。", type(dataset).__name__],
        [6, "dataset.data_dir", "ProsumerDataset 固定读取 data_dir / processed/prosumer。", str(dataset.data_dir)],
        [7, "dataset.get_episode(0)['signals']['load'].shape", "输出给环境的是按 episode 切好的数组，不是 registry 返回 DataFrame。", signals["load"].shape],
        [8, "dataset.get_episode(0)['signals']['wholesale_price'].shape", "批发电价是一维时间序列，长度等于 episode_length。", signals["wholesale_price"].shape],
    ]

show_steps(
    "5. build_dataset(train) 的逐句演算",
    "data_dir=临时 data, episode_limit=4, train_window_days=2, n_agents=2",
    rows,
    {
        "is_prosumer_dataset": isinstance(dataset, ProsumerDataset),
        "dataset_data_dir": str(dataset.data_dir),
        "expected_data_dir": str(expected_data_dir),
        "num_episodes": dataset.num_episodes(),
        "signal_shapes": {name: value.shape for name, value in signals.items()},
    },
)

assert isinstance(dataset, ProsumerDataset)
assert dataset.data_dir == expected_data_dir
assert dataset.episode_length == 8
assert signals["load"].shape == (8, cfg.env.num_agents)
assert signals["pv"].shape == (8, cfg.env.num_agents)
assert signals["wholesale_price"].shape == (8,)


5. build_dataset(train) 的逐句演算
输入：data_dir=临时 data, episode_limit=4, train_window_days=2, n_agents=2


,步骤,对应源码/表达式,怎么算,结果
0,1,data_dir = Path(cfg.data.data_dir or default_data),cfg.data.data_dir 已由 make_smoke_config 指向临时 data 目录。,C:\Users\Admin\AppData\Local\Temp\tmpa5uq3i5l\data
1,2,"selected_year, start_date, end_date = _resolve_split_dates(cfg, 'train')",训练 split 选择 train_year；默认没有显式日期边界。,"(2019, None, None)"
2,3,"window_spec = resolve_dataset_window_spec(cfg, 'train')","复用第 3 节数学：episode_length=4*2=8, stride=4*1=4。","{'episode_length': 8, 'window_stride_steps': 4, 'window_strategy': 'rolling_window', 'window_days': 2, 'window_stride_days': 1, 'base_ep..."
3,4,history_warmup_steps = forecast.history_window if forecast.type == 'lstm' else 0,当前 forecast.type='perfect'，不是 LSTM 主线则为 0。,0
4,5,"ProsumerDataset(data_dir=..., episode_length=..., n_agents=...)",把所有 cfg 字段整理成 ProsumerDataset 构造参数。,ProsumerDataset
5,6,dataset.data_dir,ProsumerDataset 固定读取 data_dir / processed/prosumer。,C:\Users\Admin\AppData\Local\Temp\tmpa5uq3i5l\data\processed\prosumer
6,7,dataset.get_episode(0)['signals']['load'].shape,输出给环境的是按 episode 切好的数组，不是 registry 返回 DataFrame。,"(8, 2)"
7,8,dataset.get_episode(0)['signals']['wholesale_price'].shape,批发电价是一维时间序列，长度等于 episode_length。,"(8,)"


最终输出：


{'is_prosumer_dataset': True,
 'dataset_data_dir': 'C:\\Users\\Admin\\AppData\\Local\\Temp\\tmpa5uq3i5l\\data\\processed\\prosumer',
 'expected_data_dir': 'C:\\Users\\Admin\\AppData\\Local\\Temp\\tmpa5uq3i5l\\data\\processed\\prosumer',
 'num_episodes': 2,
 'signal_shapes': {'wholesale_price': (8,),
  'load': (8, 2),
  'pv': (8, 2),
  'load_household': (8, 2),
  'load_heatpump': (8, 2)}}

## 6. `build_dataset(..., override_start_date, override_end_date)`：shared-data 场景下覆盖日期

`build_dataset` 暴露 override 参数，是为了让 shared-data manifest 这类入口能用“缓存里实际声明的 split 日期”，而不是盲目使用当前 cfg 里的日期。

In [7]:
with TemporaryDirectory() as temp_dir:
    cfg = make_registry_demo_cfg(temp_dir, episode_limit=4, train_window_days=2, window_stride_days=1)
    cfg.data.test_start_date = "2020-04-01"
    cfg.data.test_end_date = "2020-04-15"

    dataset = build_dataset(
        cfg,
        mode="test",
        override_start_date="2020-01-01",
        override_end_date="2020-01-01",
    )
    episode = dataset.get_episode(0)

    rows = [
        [1, "_resolve_split_dates(cfg, 'test')", "先按 test split 选 test_year=2020。", dataset.year],
        [2, "start_date = cfg.data.test_start_date", "cfg 里原本是 2020-04-01。", "2020-04-01"],
        [3, "override_start_date is not _USE_CFG_VALUE", "调用方传了 '2020-01-01'，不是哨兵值，所以覆盖 cfg。", True],
        [4, "start_date = _normalized_optional_date(override_start_date)", "覆盖后的 start_date 是 2020-01-01。", dataset.start_date.isoformat()],
        [5, "override_end_date 同理", "cfg 里原本是 2020-04-15，调用方覆盖成 2020-01-01。", dataset.end_date.isoformat()],
        [6, "resolve_dataset_window_spec(cfg, 'test')", "测试集不使用 train_window_days，所以 episode_length=base_episode_length=4。", dataset.episode_length],
        [7, "ProsumerDataset(..., split='test')", "最终构建 test split dataset，并只保留 2020-01-01 当天的 rows。", dataset.split],
        [8, "dataset.get_episode(0)['meta']['timestamps']", "第一个 episode 的时间戳落在 override 后的日期窗口内。", episode["meta"]["timestamps"][:2]],
    ]

show_steps(
    "6. build_dataset(test override) 的逐句演算",
    "cfg test range=2020-04-01..2020-04-15，但 override=2020-01-01..2020-01-01",
    rows,
    {
        "split": dataset.split,
        "year": dataset.year,
        "date_range": (dataset.start_date.isoformat(), dataset.end_date.isoformat()),
        "episode_length": dataset.episode_length,
        "num_episodes": dataset.num_episodes(),
    },
)

assert dataset.split == "test"
assert dataset.year == cfg.data.test_year
assert dataset.start_date.isoformat() == "2020-01-01"
assert dataset.end_date.isoformat() == "2020-01-01"
assert dataset.episode_length == 4
assert dataset.num_episodes() > 0


6. build_dataset(test override) 的逐句演算
输入：cfg test range=2020-04-01..2020-04-15，但 override=2020-01-01..2020-01-01


,步骤,对应源码/表达式,怎么算,结果
0,1,"_resolve_split_dates(cfg, 'test')",先按 test split 选 test_year=2020。,2020
1,2,start_date = cfg.data.test_start_date,cfg 里原本是 2020-04-01。,2020-04-01
2,3,override_start_date is not _USE_CFG_VALUE,调用方传了 '2020-01-01'，不是哨兵值，所以覆盖 cfg。,True
3,4,start_date = _normalized_optional_date(override_start_date),覆盖后的 start_date 是 2020-01-01。,2020-01-01
4,5,override_end_date 同理,cfg 里原本是 2020-04-15，调用方覆盖成 2020-01-01。,2020-01-01
5,6,"resolve_dataset_window_spec(cfg, 'test')",测试集不使用 train_window_days，所以 episode_length=base_episode_length=4。,4
6,7,"ProsumerDataset(..., split='test')",最终构建 test split dataset，并只保留 2020-01-01 当天的 rows。,test
7,8,dataset.get_episode(0)['meta']['timestamps'],第一个 episode 的时间戳落在 override 后的日期窗口内。,"[2020-01-01 00:00:00+01:00, 2020-01-01 00:15:00+01:00]"


最终输出：


{'split': 'test',
 'year': 2020,
 'date_range': ('2020-01-01', '2020-01-01'),
 'episode_length': 4,
 'num_episodes': 3}

## 总结

现在的 `registry.py` 可以理解成一个很窄的主线适配层：

1. 解析可选日期。
2. 解析 train/test split 的年份和日期边界。
3. 计算 train/test episode window 规格。
4. 把 `ExperimentConfig` 明确翻译成 `ProsumerDataset(...)`。

它不再是动态数据集注册中心；当前项目主线就是 `ProsumerDataset`。